In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/squid-dataset/train.json


In [4]:
!pip install rank_bm25
import json, math
import pandas as pd, numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [5]:


# ---------- Load SQuAD JSON ----------
file_path = '/kaggle/input/squid-dataset/train.json'

def convert_squad_data_to_dataframe(file_path, cnt=0):
    with open(file_path, 'r') as f:
        data = json.load(f)['data']
        rows = []
        for article in data:
            for paragraph in article['paragraphs']:
                ctx = paragraph['context']
                for qa in paragraph['qas']:
                    q = qa['question']
                    a = qa['answers'] if not qa.get('is_impossible', False) else []
                    if a:
                        for ans in a:
                            rows.append({'context': ctx, 'question': q, 'answer': ans['text'], 'c_id': cnt})
                    else:
                        rows.append({'context': ctx, 'question': q, 'answer': '', 'c_id': cnt})
                cnt += 1
        return pd.DataFrame(rows)

data = convert_squad_data_to_dataframe(file_path)

# ---------- Preprocess ----------
documents = data[['context', 'c_id']].drop_duplicates().reset_index(drop=True)
c_id_array = documents['c_id'].to_numpy()

# 🔥 Fast tokenizer
def fast_tokenize(text):
    return [w for w in text.lower().split() if w not in ENGLISH_STOP_WORDS]

documents['tokenized_context'] = documents['context'].map(fast_tokenize)
bm25 = BM25Okapi(documents['tokenized_context'].tolist())

# Mapping questions to gold context ID
questionContext = dict(zip(data['question'], data['c_id']))
questions = list(questionContext.keys())
tokenized_questions = [fast_tokenize(q) for q in questions]





In [6]:
# ---------- Metrics ----------
def dcg(scores):
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(scores))

def compute_metrics_bm25(questions, questionContext, tokenized_questions, bm25, k=10, max_samples=2000):
    total_correct, reciprocal_ranks, ndcg_scores = 0, [], []
    precision_scores, recall_scores, mar_scores = [], [], []

    for i in tqdm(range(min(max_samples, len(questions)))):
        q_tok = tokenized_questions[i]
        scores = bm25.get_scores(q_tok)
        top_k = np.argpartition(-scores, k)[:k]
        top_k = top_k[np.argsort(scores[top_k])[::-1]]
        preds = c_id_array[top_k]
        actual = questionContext[questions[i]]

        relevance = [1 if p == actual else 0 for p in preds]

        if actual in preds:
            rank = preds.tolist().index(actual)
            total_correct += 1
            reciprocal_ranks.append(1.0 / (rank + 1))
            precision_scores.append(1.0 / (rank + 1))
            recall_scores.append(1.0)
        else:
            reciprocal_ranks.append(0.0)
            precision_scores.append(0.0)
            recall_scores.append(0.0)

        ndcg_scores.append(dcg(relevance) / dcg(sorted(relevance, reverse=True)) if sum(relevance) else 0.0)
        mar_scores.append(sum(relevance) / k)

    num_samples = min(len(questions), max_samples)
    return {
        f"Accuracy@{k}": (total_correct / num_samples) * 100,
        f"MRR@{k}": np.mean(reciprocal_ranks),
        f"NDCG@{k}": np.mean(ndcg_scores),
        f"Recall@{k}": np.mean(recall_scores),
        f"Precision@{k}": np.mean(precision_scores),
        f"MAR@{k}": np.mean(mar_scores)
    }

In [7]:
# ---------- Run Evaluation ----------
final_result = []
for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200, 250, 300]:
    metrics = compute_metrics_bm25(questions, questionContext, tokenized_questions, bm25, k=k, max_samples=86769)
    final_result.append(metrics)
    print(f"\n🔍 Top-{k} Evaluation Metrics:")
    for key, val in metrics.items():
        print(f"{key}: {val:.4f}")


100%|██████████| 86769/86769 [37:13<00:00, 38.85it/s]


🔍 Top-100 Evaluation Metrics:
Accuracy@100: 85.4729
MRR@100: 0.5667
NDCG@100: 0.6274
Recall@100: 0.8547
Precision@100: 0.5667
MAR@100: 0.0085
